In [ ]:
!pip install -q jiwer
!pip install -q torch
!pip install -q torchvision
!pip install torchinfo

!pip install -q fastwer jiwer

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from jiwer import wer
import fastwer
from torchinfo import summary
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler

In [ ]:
from pathlib import Path

data_dir = Path(os.environ.get("OCR_DATA_DIR", "./data/VietOCR"))
image_dir = data_dir / "InkData_line_processed"
train_annotations = data_dir / "train_line_annotation.txt"
test_annotations = data_dir / "test_line_annotation.txt"

missing_paths = [str(path) for path in (train_annotations, test_annotations) if not path.is_file()]
if not image_dir.is_dir():
    missing_paths.append(str(image_dir))
if missing_paths:
    raise FileNotFoundError(
        "Missing local training data: " + ", ".join(missing_paths)
        + ". Request HANDS-VNOnDB access at https://zenodo.org/records/23864366"
        + " and follow the local preparation instructions in README.md."
    )

columns = ["id", "label"]
train_line_data = pd.read_csv(train_annotations, sep="\t", header=None, names=columns)
test_line_data = pd.read_csv(test_annotations, sep="\t", header=None, names=columns)

# Vocab

In [ ]:
class Vocab:
    def __init__(self, chars):
        self.blank = 0
        self.pad = 1
        self.go = 2
        self.eos = 3
        self.mask_token = 4
        self.chars = chars
        self.c2i = {c: i+5 for i, c in enumerate(chars)}
        self.i2c = {i+5: c for i, c in enumerate(chars)}
        self.i2c[0] = '<blank>'
        self.i2c[1] = '<pad>'
        self.i2c[2] = '<sos>'
        self.i2c[3] = '<eos>'
        self.i2c[4] = '*'

    def encode(self, chars):
        return [self.c2i[c] for c in chars]

    def decode(self, ids):
        if isinstance(ids, np.ndarray):
            ids = ids.tolist()
        result = []
        prev = None
        for i in ids:
            if i == self.blank:
                continue
            if i != prev and i > 4:
                result.append(i)
            prev = i
        return ''.join([self.i2c[i] for i in result if i in self.i2c])

    def __len__(self):
        return len(self.c2i) + 5

    def batch_decode(self, arr):
        return [self.decode(ids) for ids in arr]

    def __str__(self):
        return self.chars

vocab = Vocab('aAàÀảẢãÃáÁạẠăĂằẰẳẲẵẴắẮặẶâÂầẦẩẨẫẪấẤậẬbBcCdDđĐeEèÈẻẺẽẼéÉẹẸêÊềỀểỂễỄếẾệỆfFgGhHiIìÌỉỈĩĨíÍịỊjJkKlLmMnNoOòÒỏỎõÕóÓọỌôÔồỒổỔỗỖốỐộỘơƠờỜởỞỡỠớỚợỢpPqQrRsStTuUùÙủỦũŨúÚụỤưƯừỪửỬữỮứỨựỰvVwWxXyYỳỲỷỶỹỸýÝỵỴzZ0123456789!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~ ')
vocab_size = len(vocab)

# Image Preprocessing

In [ ]:
def process(img_path):
    img = cv2.imread(img_path, 0)
    if img is None:
        return None
    kernel = np.ones((1, 1), np.uint8)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 5)
    img = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)
    img = cv2.dilate(img, kernel, iterations=1)
    return img

def preprocess_image(image_path, label_len, img_h=64, char_w=32, max_label_len=165):
    img = process(image_path)
    if img is None:
        return None
    # label_len = min(label_len, max_label_len)
    img_w = label_len * char_w
    img = cv2.resize(img, (img_w, img_h), interpolation=cv2.INTER_AREA)
    img = np.expand_dims(img, axis=0) / 255.0
    return torch.tensor(img, dtype=torch.float32)

# Dataset and Collate Function

In [ ]:
# Dataset
class HandwrittenDataset(Dataset):
    def __init__(self, data, image_dir, vocab):
        self.data = data.dropna(subset=['label'])
        self.data = self.data[self.data['label'].str.strip() != '']
        self.image_dir = image_dir
        self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def clean_text(self, text):
        allowed_chars = set(self.vocab.c2i.keys())
        return ''.join([char if char in allowed_chars else '' for char in text])

    def encode_to_labels(self, label, max_length=None):
        label = self.clean_text(label)
        label_indices = self.vocab.encode(label)
        if max_length is not None and len(label_indices) < max_length:
            label_indices += [self.vocab.pad] * (max_length - len(label_indices))
        return label_indices

    def __getitem__(self, idx):
        img_id = self.data.iloc[idx]['id']
        label = self.data.iloc[idx]['label']
        label_len = len(label)
        image_path = os.path.join(self.image_dir, f"{img_id}")
        if not os.path.exists(image_path):
            return self.__getitem__((idx + 1) % len(self))
        image = preprocess_image(image_path, label_len)
        if image is None:
            return self.__getitem__((idx + 1) % len(self))
        label_indices = self.encode_to_labels(label)
        return image, torch.tensor(label_indices, dtype=torch.long), len(label_indices)

    def show_samples(self, idx=None):
        if idx is None:
            idx = np.random.randint(0, len(self))
        image, label_tensor, label_len = self[idx]
        image = image.squeeze(0).numpy() * 255
        decoded_label = self.vocab.decode(label_tensor.tolist())
        print(f"Sample {idx} - Label: {decoded_label}, Length: {label_len}")
        plt.figure(figsize=(10, 2))
        plt.imshow(image, cmap='gray')
        plt.axis('off')
        plt.show()

# Collate function
def handwritten_collate_fn(batch):
    images, labels, label_lengths = zip(*batch)
    max_width = max(img.shape[2] for img in images)
    max_label_length = max(label_lengths)
    padded_images = []
    for img in images:
        _, h, w = img.shape
        if w < max_width:
            pad_size = max_width - w
            padding = torch.full((1, h, pad_size), fill_value=1.0)
            padded_img = torch.cat([img, padding], dim=2)
        else:
            padded_img = img
        padded_images.append(padded_img)
    images_tensor = torch.stack(padded_images)
    labels_padded = torch.zeros(len(labels), max_label_length, dtype=torch.long)
    for i, label in enumerate(labels):
        labels_padded[i, :label_lengths[i]] = label
    label_lengths_tensor = torch.tensor(label_lengths, dtype=torch.long)
    return images_tensor, labels_padded, label_lengths_tensor

In [ ]:
train_dataset = HandwrittenDataset(train_line_data, image_dir, vocab)
test_dataset = HandwrittenDataset(test_line_data, image_dir, vocab)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=handwritten_collate_fn, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=handwritten_collate_fn, num_workers=4, pin_memory=True)

# Model

In [ ]:
class CRNN_RNN(nn.Module):
    def __init__(self, vocab_size, img_channels=1, rnn_hidden=512):
        super(CRNN_RNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(img_channels, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2), nn.Dropout(0.1),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2), nn.Dropout(0.1),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2, 2), nn.Dropout(0.1),
            nn.Conv2d(256, 256, 3, stride=(2, 2), padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout(0.1),
            nn.Conv2d(256, 512, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.Dropout(0.1),
            nn.Conv2d(512, 512, 3, stride=(2, 1), padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Conv2d(512, 1024, 3, padding=1), nn.BatchNorm2d(1024), nn.ReLU(), nn.Dropout(0.2)
        )
        self.rnn = nn.RNN(input_size=1024, hidden_size=rnn_hidden, num_layers=2, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(rnn_hidden, vocab_size)

    def forward(self, x):
        features = self.cnn(x)
        b, c, h, w = features.size()
        features = features.permute(0, 3, 1, 2).contiguous().view(b, w, c * h)
        rnn_out = self.rnn(features)[0]
        rnn_out = self.dropout(rnn_out)
        output = self.classifier(rnn_out)
        return output.permute(1, 0, 2)

# Evaluate and Compute CTC_loss (Val_loss) Functions

In [ ]:
def evaluate(model, dataloader, vocab, device):
    model.eval()
    total_correct_chars = 0
    total_chars = 0
    total_correct_sequences = 0
    total_sequences = 0
    total_correct_batch = 0
    total_batch_chars = 0
    total_wer = 0
    total_cer = 0
    total_samples = 0
    with torch.no_grad():
        for images, labels_padded, label_lengths in dataloader:
            images = images.to(device)
            labels_padded = labels_padded.to(device)
            with autocast():
                outputs = model(images)
                log_probs = F.log_softmax(outputs, dim=2)
                predicted_indices = log_probs.argmax(dim=2)
            predicted_texts = vocab.batch_decode(predicted_indices.permute(1, 0).cpu().numpy())
            target_texts = [vocab.decode(labels_padded[i][:label_lengths[i]].cpu().numpy()) for i in range(len(labels_padded))]
            for pred, target, label_len in zip(predicted_texts, target_texts, label_lengths):
                pred = pred.replace('*', '')
                if not pred or not target:
                    continue
                # Accuracy full sequence
                if pred == target:
                    total_correct_sequences += 1
                total_sequences += 1
                # Accuracy per character (tinh tren toan bo chuoi muc tieu)
                for i in range(len(target)):
                    if i < len(pred) and pred[i] == target[i]:
                        total_correct_chars += 1
                    total_chars += 1
                # Batch-level character accuracy (chi tinh tren phan giao nhau)
                for i in range(min(len(pred), len(target))):
                    if pred[i] == target[i]:
                        total_correct_batch += 1
                    total_batch_chars += 1
                # WER and CER
                try:
                    wer_value = wer(target, pred)
                    cer_value = fastwer.score_sent(pred, target, char_level=True)
                    total_wer += wer_value
                    total_cer += cer_value
                    total_samples += 1
                except ValueError:
                    continue
    acc = total_correct_batch / total_batch_chars if total_batch_chars > 0 else 0.0
    acc_full_seq = total_correct_sequences / total_sequences if total_sequences > 0 else 0.0
    acc_per_char = total_correct_chars / total_chars if total_chars > 0 else 0.0
    avg_wer = total_wer / total_samples if total_samples > 0 else 0.0
    avg_cer = total_cer / total_samples if total_samples > 0 else 0.0
    return acc, acc_full_seq, acc_per_char, avg_wer, avg_cer

def compute_val_loss(model, dataloader, ctc_loss_fn, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, labels_padded, label_lengths in dataloader:
            images = images.to(device)
            labels_padded = labels_padded.to(device)
            with autocast():
                outputs = model(images)
                log_probs = F.log_softmax(outputs, dim=2)
                input_lengths = torch.full(size=(images.size(0),), fill_value=log_probs.size(0), dtype=torch.long).to(device)
                targets = []
                for i in range(len(labels_padded)):
                    targets.extend(labels_padded[i][:label_lengths[i]].tolist())
                targets = torch.tensor(targets, dtype=torch.long).to(device)
                loss = ctc_loss_fn(log_probs, targets, input_lengths, label_lengths)
            total_loss += loss.item()
    return total_loss / len(dataloader)

# Training settings

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, verbose=False):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.verbose = verbose

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

def train_one_epoch(model, dataloader, optimizer, ctc_loss_fn, device, scaler):
    model.train()
    total_loss = 0
    for images, labels_padded, label_lengths in dataloader:
        images = images.to(device)
        labels_padded = labels_padded.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            log_probs = F.log_softmax(outputs, dim=2)
            input_lengths = torch.full(size=(images.size(0),), fill_value=log_probs.size(0), dtype=torch.long).to(device)
            targets = []
            for i in range(len(labels_padded)):
                targets.extend(labels_padded[i][:label_lengths[i]].tolist())
            targets = torch.tensor(targets, dtype=torch.long).to(device)
            loss = ctc_loss_fn(log_probs, targets, input_lengths, label_lengths)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [ ]:
# Main
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN_RNN(vocab_size=len(vocab)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
ctc_loss_fn = nn.CTCLoss(blank=vocab.blank, zero_infinity=True)
scaler = GradScaler()
early_stopping = EarlyStopping(patience=70, verbose=True)

In [ ]:
checkpoint_dir = './trained_model'
os.makedirs(checkpoint_dir, exist_ok=True)
best_acc_per_char = 0.0
best_acc_full_seq = 0.0
best_model_path = os.path.join(checkpoint_dir, 'best_model_line.pth')

# Training

In [ ]:
EPOCHS = 100
train_losses, val_losses, val_accs, val_acc_full_seqs, val_acc_per_chars, val_wers, val_cers = [], [], [], [], [], [], []

for epoch in range(EPOCHS):
    # torch.cuda.empty_cache()
    train_loss = train_one_epoch(model, train_loader, optimizer, ctc_loss_fn, device, scaler)
    train_losses.append(train_loss)
    val_loss = compute_val_loss(model, test_loader, ctc_loss_fn, device)
    val_losses.append(val_loss)
    acc, acc_full_seq, acc_per_char, val_wer, val_cer = evaluate(model, test_loader, vocab, device)
    val_accs.append(acc)
    val_acc_full_seqs.append(acc_full_seq)
    val_acc_per_chars.append(acc_per_char)
    val_wers.append(val_wer)
    val_cers.append(val_cer)
    
    if acc_per_char > best_acc_per_char or acc_full_seq > best_acc_full_seq:
        best_acc_per_char = max(acc_per_char, best_acc_per_char)
        best_acc_full_seq = max(acc_full_seq, best_acc_full_seq)
        torch.save(model.state_dict(), best_model_path)
        print(f"Epoch {epoch+1}: Saved best model with Acc Per Char: {acc_per_char:.3f}, Acc Full Seq: {acc_full_seq:.3f}")
    
    scheduler.step(val_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
          f"Val Acc: {acc:.3f}, Val Acc Full Seq: {acc_full_seq:.3f}, Val Acc Per Char: {acc_per_char:.3f}, "
          f"Val WER: {val_wer:.3f}, Val CER: {val_cer:.3f}")
    
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(val_accs, label='Validation Accuracy')
plt.plot(val_acc_full_seqs, label='Validation Acc Full Seq')
plt.plot(val_acc_per_chars, label='Validation Acc Per Char')
plt.plot(val_wers, label='Validation WER')
plt.plot(val_cers, label='Validation CER')
plt.xlabel('Epoch')
plt.ylabel('Metrics')
plt.legend()
plt.tight_layout()
plt.savefig('training_metrics.png')

# Testing on Test Dataset

In [ ]:
test_acc, test_acc_full_seq, test_acc_per_char, test_wer, test_cer = evaluate(model, test_loader, vocab, device)
print(f"Test Acc: {test_acc:.3f}, Test Acc Full Seq: {test_acc_full_seq:.3f}, Test Acc Per Char: {test_acc_per_char:.3f}, "f"Test WER: {test_wer:.3f}, Test CER: {test_cer:.3f}")